<a href="https://colab.research.google.com/github/UniVR-DH/DKR-course/blob/main/L18-advanced/SecurityLakeAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Enterprise Security Data Lake: Schema & Telemetry Reference

This dataset simulates a production cloud-native enterprise application mesh. It architecture is optimized for forensic threat hunting, detection engineering baseline analysis, and compliance verification. It isolates logical asset inventory states, raw execution logs, and centralized security telemetry into specialized atomic tables.

---

## 1. Asset Inventory (`asset_inventory.csv`)

**Logical Context:** This table serves as the primary configuration management asset registry. It records a point-in-time snapshot of every running system instance or container node deployed across the enterprise infrastructure. It maps environmental boundaries, network identities, and specific software versions.

* **Primary Key:** `instance_id`
* **Unique Constraint:** `assigned_private_ip`

### Column Specifications

| Column Name | Data Type | Sample Value | Description / Validation Rules |
| --- | --- | --- | --- |
| `instance_id` | String (Alpha-Numeric) | `i-api-1001` | **Primary Key.** Unique infrastructure hardware or container instance identifier. |
| `assigned_private_ip` | String (IPv4 Format) | `10.0.1.11` | The internal private IPv4 address assigned to the host within its virtual network mesh. Must be unique per instance. |
| `service_name` | String (Categorical) | `api-gateway` | The logical microservice function running on this instance (e.g., `api-gateway`, `auth-service`, `customer-db`). |
| `software_component` | String (Text) | `nginx` | The primary underlying application framework or binary serving requests (e.g., `nginx`, `spring-boot`, `postgresql`). |
| `software_version` | String (SemVer) | `1.23.1` | The exact semantic version (`Major.Minor.Patch`) of the running software component, critical for matching against vulnerability databases. |
| `network_zone` | String (Enum) | `DMZ_Edge` | Architectural network exposure boundaries. Allowed values: `DMZ_Edge` (public-facing), `Internal_Core` (private app mesh), `Data_Vault` (isolated database layer). |
| `subnet_cidr` | String (CIDR notation) | `10.0.1.0/24` | The explicit network block containing the instance, detailing the sub-network range boundaries. |

---

## 2. Network Access Logs (`network_access_logs.csv`)

**Logical Context:** High-volume raw operational telemetry streaming directly from system proxies, API routers, and communication interfaces. It captures every transactional hook, execution latency proxy, and transaction footprint crossing between nodes or entering from the public internet.

* **Primary Key:** `log_id`
* **Foreign Key Dependencies:** `dst_ip` $\rightarrow$ `asset_inventory.csv(assigned_private_ip)`

### Column Specifications

| Column Name | Data Type | Sample Value | Description / Validation Rules |
| --- | --- | --- | --- |
| `log_id` | String (Unique ID) | `log_att-001` | **Primary Key.** A completely unique transaction tracker for each log entry. |
| `timestamp` | DateTime (ISO 8601) | `2026-05-18T10:15:00.000` | The exact millisecond-precision moment the communication network packet was parsed by the destination node. |
| `trace_id` | String (Alphanumeric) | `tr-attack-9999` | Distributed tracing correlation token propagated across downstream headers. Allows a multi-tier request chain to be reconstructed step-by-step. |
| `src_ip` | String (IPv4 Format) | `185.220.101.5` | The originating IP address of the traffic. Can be an untrusted public internet address or an internal instance private IP during pivoting. |
| `dst_ip` | String (IPv4 Format) | `10.0.1.11` | **Foreign Key.** The receiving internal destination private IP. Matches directly to a node in the asset ledger. |
| `dst_port` | Integer | `443` | The target TCP network port where the service handler intercepts requests (e.g., `80`, `443`, `8080`, `5432`). |
| `dst_service_name` | String (Categorical) | `api-gateway` | The logical microservice function handling the request (e.g., `api-gateway`, `auth-service`, `customer-db`). |
| `request_path` | String (URI / Text) | `/api/v1/admin/upload` | The precise web endpoint, routing path, or low-level connection payload action executed against the application target. |
| `http_status` | Integer (HTTP Code) | `500` | The network layer response code (e.g., `200` OK, `401` Unauthorized, `403` Forbidden, `500` Internal Server Error). Non-web layer events use database native response maps. |
| `payload_bytes` | Integer (Positive) | `8500` | The volume size of the response or transit packet payload measured in bytes. |

---

## 3. SIEM Incidents (`siem_incidents.csv`)

**Logical Context:** Security intelligence notifications and anomaly triggers bubbled up from real-time Endpoint Detection and Response (EDR) agents, Web Application Firewalls (WAF), or Security Information and Event Management (SIEM) systems. It registers severe alert exceptions contextually isolated down to precise application routes.

* **Primary Key:** `incident_id`
* **Foreign Key Dependencies:** `target_instance_id` $\rightarrow$ `asset_inventory.csv(instance_id)`, `triggering_log_id` $\rightarrow$ `network_access_logs.csv(log_id)`

### Column Specifications

| Column Name | Data Type | Sample Value | Description / Validation Rules |
| --- | --- | --- | --- |
| `incident_id` | String (Alpha-Numeric) | `INC-2026-001` | **Primary Key.** Unique tracking number generated by the centralized incident response dashboard. |
| `alert_timestamp` | DateTime (ISO 8601) | `2026-05-18T10:15:02.000` | The formal timestamp identifying when the alert logic or detection rule flagged the anomalous runtime activity. |
| `target_instance_id` | String (Alpha-Numeric) | `i-api-1001` | **Foreign Key.** Pinpoints exactly which system instance host was targeted by the exploit. |
| `triggering_log_id` | String (Unique ID) | `log_att-001` | **Foreign Key.** The smoking gun token linking this security exception directly back to the atomic network connection that triggered the rule. |
| `exploited_component` | String (Text) | `nginx` | Explicitly verifies which local software daemon or framework layer processing the request fell victim to the incident conditions. |
| `targeted_endpoint` | String (URI / Text) | `/api/v1/admin/upload` | The application route or active port endpoint where malicious behaviors or exploits were executed. |
| `incident_type` | String (Categorical) | `Remote_Code_Execution` | Classification taxonomy for the malicious behavior. Common fields: `Remote_Code_Execution`, `Anomalous_Internal_Portscan`, `Credential_Dump`. |
| `severity` | String (Enum) | `CRITICAL` | Calculated impact priority rating. Restrained strictly to: `LOW`, `MEDIUM`, `HIGH`, or `CRITICAL`. |
| `cve_id` | String (CVE Format) | `CVE-2021-44228` | The standardized Common Vulnerabilities and Exposures code tracking the application vulnerability. If the alert is an behavioral attack type, defaults to `NONE`. |
| `mitigation_action` | String (Categorical) | `Host_Isolated` | Operational isolation or blocking action applied to the infrastructure on alert generation (e.g., `None`, `Host_Isolated`, `Traffic_Blocked`). |

---

## 4. Service Permissions (`service_permissions.csv`)

**Logical Context:** The architectural authorization manifest ("The Golden State Policy"). This table acts as a structural reference map outlining what paths of server-to-server microservice communication are formally authorized in the company code. Any transactional event logged in network telemetry that is not registered here represents an immediate logical baseline violation.

* **Primary Key:** None (Relies entirely on the natural composite key combination of `caller_service` + `target_service`)

### Column Specifications

| Column Name | Data Type | Sample Value | Description / Validation Rules |
| --- | --- | --- | --- |
| `caller_service` | String (Categorical) | `api-gateway` | The authorized initiating microservice allowed to execute outbound API queries or database connections. |
| `target_service` | String (Categorical) | `auth-service` | The designated receiver microservice permitted to ingest and execute the inbound request chain. |
| `authorization_scope` | String (Categorical) | `read-write` | The permission clearance assigned to the architectural path transaction channel (e.g., `read-only`, `read-write`, `admin`). |

---

## Analytical Correlation Framework (How They Interlock)

To run security validations across this data lake, data engineers merge data sets dynamically using the following operational relational lookup loops:



```
┌─────────────────────────────────────────────────────────────────────────────┐
│                         network_access_logs                                 │
│                    log_id · dst_ip · trace_id · src_ip                      │
└─────────────────────┬───────────────────────────────┬───────────────────────┘
                      │                               │
              JOIN via dst_ip                 JOIN via log_id
                      │                               │
                      ▼                               ▼
        ┌───────────────────────┐       ┌──────────────────────────┐
        │    asset_inventory    │       │      siem_incidents      │
        │ instance_id · version │       │   cve_id · severity      │
        │ service_name · zone   │       │    exploited_component   │
        └───────────┬───────────┘       └────────────┬─────────────┘
                    │                                │
          compare service_name              compare software_component
                    │                                │
                    └──────────────┬─────────────────┘
                                   ▼
                    ┌──────────────────────────────┐
                    │     service_permissions      │
                    │ caller · target · auth_scope │
                    └──────────────────────────────┘

```

1. **Vulnerability & Blast Radius Assessment:** By filtering an incident on `cve_id`, you trace back to the `exploited_component` and `software_version`. You can instantly join this back against the `asset_inventory` to detect every single deployment running the matching software component across different network zones.
2. **Downstream Threat Hunting / Lateral Movement Analysis:** When a `siem_incident` is raised, you can query its `triggering_log_id` inside `network_access_logs`. From there, extracting that row's unique `trace_id` unlocks the timeline of every internal hop sharing that token—revealing exactly where the attacker pivoted after hitting the entry barrier.
3. **Policy Compliance Audits:** By merging your daily raw traffic paths (`network_access_logs` connected to `asset_inventory` details) against the authorized blueprint rules in `service_permissions`, you can dynamically catch anomalous direct lines of communication that circumvent the infrastructure design rules (such as a public gateway attempting to communicate straight to a private data vault).

In [ ]:
%pip install grafeo

In [4]:
import pandas as pd

# Load asset_inventory.csv
asset_inventory_df = pd.read_csv('asset_inventory.csv')
print('asset_inventory.csv:')
display(asset_inventory_df.head(4))
print('\n')

# Load network_access_logs.csv
network_access_logs_df = pd.read_csv('network_access_logs.csv')
print('network_access_logs.csv:')
display(network_access_logs_df.head(4))
print('\n')

# Load siem_incidents.csv
siem_incidents_df = pd.read_csv('siem_incidents.csv')
print('siem_incidents.csv:')
display(siem_incidents_df.head(4))
print('\n')

# Load service_permissions.csv
service_permissions_df = pd.read_csv('service_permissions.csv')
print('service_permissions.csv:')
display(service_permissions_df.head(4))
print('\n')

asset_inventory.csv:


,instance_id,assigned_private_ip,service_name,software_component,software_version,network_zone,subnet_cidr
0,i-api-1001,10.0.1.11,api-gateway,nginx,1.23.1,DMZ_Edge,10.0.1.0/24
1,i-api-1002,10.0.1.12,api-gateway,nginx,1.23.1,DMZ_Edge,10.0.1.0/24
2,i-auth-2001,10.0.2.11,auth-service,spring-boot,2.7.3,Internal_Core,10.0.2.0/24
3,i-auth-2002,10.0.2.12,auth-service,spring-boot,2.7.3,Internal_Core,10.0.2.0/24




network_access_logs.csv:


,log_id,timestamp,trace_id,src_ip,dst_ip,dst_port,dst_service_name,request_path,http_status,payload_bytes
0,log-0356,2026-05-18T08:00:00.579,tr-47615,10.0.0.1,10.0.3.12,8080,order-svc,/health,200,104
1,log-0229,2026-05-18T08:00:18.391,tr-36336,10.0.3.12,10.0.5.11,9200,log-aggregator,/search,200,3372
2,log-0298,2026-05-18T08:00:22.582,tr-74527,10.0.3.12,10.0.5.11,9200,log-aggregator,/bulk,200,3423
3,log-0140,2026-05-18T08:00:24.451,tr-17172,193.32.127.92,10.0.1.12,443,api-gateway,/api/v1/login,200,1570




siem_incidents.csv:


,incident_id,alert_timestamp,target_instance_id,triggering_log_id,exploited_component,targeted_endpoint,incident_type,severity,cve_id,mitigation_action
0,INC-2026-002,2026-05-18T08:15:15.250,i-auth-2001,log-0404,spring-boot,/auth/token,Remote_Code_Execution,MEDIUM,CVE-2022-22965,NaN
1,INC-2026-003,2026-05-18T08:23:48.899,i-pay-3001,log-0405,spring-boot,/pay/charge,Credential_Dump,LOW,NONE,NaN
2,INC-2026-004,2026-05-18T09:36:01.849,i-ord-3002,log-0406,spring-boot,/orders/create,Anomalous_Internal_Portscan,LOW,NONE,NaN
3,INC-2026-001,2026-05-18T09:51:42.359,i-api-1001,log-0403,nginx,/api/v1/login,Credential_Dump,LOW,NONE,NaN




service_permissions.csv:


,caller_service,target_service,authorization_scope
0,api-gateway,auth-service,read-write
1,api-gateway,payment-svc,read-write
2,api-gateway,order-svc,read-write
3,api-gateway,log-aggregator,read-only


In [ ]:
# Initialize local store
from grafeo import GrafeoDB

# In-memory database
db = GrafeoDB()

# Or persistent on file
# db = GrafeoDB("./my-graph")

In [ ]:
# Create nodes & Edges


(empty)


In [ ]:
# Analyze the allowed service permission
# paths starting from the api-gateway

Alix knows Gus since 2020
